# Enveda CASMI 2026 — la tâche, la data, les solutions

Notebook d'orientation. Trois questions, dans l'ordre :

1. **Quelle est la tâche ?** Ce qu'un spectromètre de masse mesure, ce qu'il faut rendre,
   et comment c'est noté.
2. **Qu'est-ce qu'on a comme données ?** 2,54 M de spectres, onze bibliothèques — et une
   asymétrie entre elles que le tutoriel officiel efface en une ligne de code.
3. **Comment on résout ça ?** Les quatre familles de modèles du domaine, ce que fait
   réellement le leaderboard, et pourquoi ce n'est pas ce que le tutoriel enseigne.

Tout ce qui est chiffré ici est mesuré sur `data/raw/`, pas repris d'un notebook public.
Les synthèses écrites vivent dans [`../docs/competition.md`](../docs/competition.md) et
[`../docs/public-meta.md`](../docs/public-meta.md) ; ici on regarde les données.

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq
import matplotlib.pyplot as plt, matplotlib as mpl
import scipy.sparse as sp
from pathlib import Path

RAW = Path("../data/raw")

# Palette : les trois premiers slots de la palette de référence, validés tous-couples
# (CVD dE 9.2, vision normale 24.0 en mode clair). On n'en utilise jamais plus de trois.
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e6e5e1"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "figure.dpi": 110, "savefig.dpi": 110,
    "text.color": INK, "axes.labelcolor": INK2, "axes.edgecolor": GRID,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.labelsize": 10, "axes.titlesize": 11, "legend.fontsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False,
    "lines.linewidth": 2, "font.size": 10,
})

pd.set_option("display.width", 150, "display.max_columns", 40)
print("prêt")

---
# 1. La tâche

## 1.1 Ce qu'un spectromètre de masse mesure

Un spectromètre de masse ne « voit » pas une molécule. Il la pèse, puis il la casse et
pèse les morceaux.

En MS/MS (ou MS², « tandem »), trois étapes :

1. **Ionisation.** La molécule neutre `M` reçoit une charge, le plus souvent en captant
   un proton : on observe `[M+H]+`. D'autres *adduits* existent (`[M-H]-` en mode
   négatif, `[M+Na]+`, `[M+NH4]+`…). L'adduit compte parce qu'il décale la masse mesurée
   d'une quantité connue — c'est ainsi qu'on remonte à la masse neutre.
2. **Sélection du précurseur.** On isole les ions d'un rapport masse/charge donné : le
   `precursor_mz`. C'est le poids de la molécule entière (plus l'adduit).
3. **Fragmentation et mesure.** On accélère ces ions contre un gaz (l'énergie de
   collision, `collision_energy_ev`), ils se brisent, et on mesure le `m/z` de chaque
   fragment avec son intensité.

Le résultat est donc une liste de couples `(m/z, intensité)` : `ms2_mzs` et
`ms2_normalized_intensities`. C'est ça, un spectre.

**L'intuition clé** : les liaisons d'une molécule ne cassent pas au hasard. Le motif des
fragments est une signature de la structure — mais une signature *dégradée*, parce que
deux molécules différentes peuvent casser en morceaux de mêmes masses. D'où la
difficulté : le spectre contraint fortement la structure sans la déterminer.

Regardons un vrai spectre du jeu de test.

In [ ]:
test = pd.read_parquet(RAW / "test.parquet")
print(f"test.parquet : {len(test):,} spectres pour {test.molecule_id.nunique()} molécules")

s = test.iloc[0]
print(f"\nmolécule {s.molecule_id}, spectre {s.spectrum_id}")
print(f"  adduit            {s.adduct}   (mode {s.ionization_mode})")
print(f"  precursor_mz      {s.precursor_mz:.4f}")
print(f"  instrument        {s.instrument_type}")
print(f"  énergie collision {np.asarray(s.collision_energy_ev)} {s.collision_energy_orig_units}")
print(f"  nombre de pics    {len(s.ms2_mzs)}")

print("\nles 8 pics les plus intenses :")
order = np.argsort(s.ms2_normalized_intensities)[::-1][:8]
print(pd.DataFrame({"m/z": np.asarray(s.ms2_mzs)[order].round(4),
                    "intensité": np.asarray(s.ms2_normalized_intensities)[order].round(4)})
      .to_string(index=False))

In [ ]:
# FIGURE 1 — un spectre de masse se lit comme un peigne : une raie verticale par fragment.
mz  = np.asarray(s.ms2_mzs)
itn = np.asarray(s.ms2_normalized_intensities)
itn = itn / itn.max()

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.vlines(mz, 0, itn, color=BLUE, linewidth=0.9)
ax.axvline(s.precursor_mz, color=ORANGE, linewidth=2, zorder=0)
ax.annotate(f"précurseur {s.precursor_mz:.2f}", xy=(s.precursor_mz, 1.0),
            xytext=(-6, -2), textcoords="offset points",
            ha="right", va="top", color=ORANGE, fontsize=9, fontweight="bold")

# étiqueter seulement les pics dominants, jamais tous
for i in np.argsort(itn)[::-1][:5]:
    ax.annotate(f"{mz[i]:.3f}", xy=(mz[i], itn[i]), xytext=(0, 4),
                textcoords="offset points", ha="center", fontsize=8, color=INK2)

ax.set_xlabel("m/z"); ax.set_ylabel("intensité relative")
ax.set_title(f"Spectre MS² de la molécule {s.molecule_id} — {len(mz)} pics, {s.instrument_type}",
             loc="left", color=INK)
ax.set_ylim(0, 1.18); ax.set_xlim(0, s.precursor_mz * 1.08)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

print("Chaque raie est un fragment. La raie orange est la molécule entière : aucun fragment")
print("ne peut la dépasser, ce qui plafonne l'axe des masses. Tout le signal utile est dans")
print("la position EXACTE des raies — 4 décimales, pas 1 — parce que c'est la précision en")
print("masse qui distingue deux formules brutes voisines.")

## 1.2 Ce qu'il faut rendre

Pas une molécule : **25 molécules classées**, par ordre de confiance décroissante.

La notation est **par molécule, pas par spectre**. Le test contient 1 213 spectres pour
400 molécules — 3 en moyenne, acquis à des énergies de collision différentes. Les
spectres d'une même molécule votent donc ensemble dans une seule liste classée.

Le format du fichier est inhabituel et vaut la peine d'être vu :

In [ ]:
ss = pd.read_csv(RAW / "sample_submission.csv")
print(f"sample_submission.csv : {ss.shape[0]} lignes × {ss.shape[1]} colonnes {list(ss.columns)}")
print(f"\nune ligne = une molécule, et les 25 candidats sont UNE chaîne séparée par des ';'")
cell = ss.smiles.iloc[0]
print(f"\n  molecule_id : {ss.molecule_id.iloc[0]}")
print(f"  smiles      : {cell[:60]}...")
print(f"                -> {len(cell.split(';'))} candidats dans cette seule cellule")
print(f"\n(le sample_submission remplit tout avec 'CCO', l'éthanol : c'est un gabarit, pas une")
print(f"prédiction. Les 400 molécules sont à prédire, aucune n'est donnée.)")

# spectres par molécule, côté test
spm = test.groupby("molecule_id").size()
print(f"\nspectres par molécule de test : min {spm.min()}, médiane {spm.median():.0f}, max {spm.max()}")
print(f"énergies de collision distinctes dans le test : {test.collision_energy_orig.nunique()}")

## 1.3 La métrique : *mean reciprocal rank*

Pour chaque molécule, on prend le **rang du premier candidat correct** et on compte
`1/rang`. Si aucun des 25 n'est correct, zéro. Le score du leaderboard est la moyenne sur
les 400 molécules.

| rang du bon candidat | 1 | 2 | 3 | 4 | 5 | 10 | 25 | absent |
|---|---|---|---|---|---|---|---|---|
| points | 1.00 | 0.50 | 0.33 | 0.25 | 0.20 | 0.10 | 0.04 | 0.00 |

**Conséquence à garder en tête en permanence** : passer du rang 2 au rang 1 vaut 0.50.
Passer du rang 25 au rang 3 vaut 0.29. Tout ce qui est sous le rang 3 ne vaut
collectivement presque rien — les sondages publics mettent toute la queue à ≤ 0.002 du
score. **Le jeu, c'est le slot 1.**

Et un piège sur ce qu'est « correct » : la comparaison se fait sur l'**InChIKey-14**, le
premier bloc de l'InChIKey, qui est un hash du squelette moléculaire. Deux conséquences.

In [ ]:
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

def key14(smiles):
    m = Chem.MolFromSmiles(smiles)
    return None if m is None else Chem.MolToInchiKey(m).split("-")[0]

print("=== 1. la stéréochimie est IGNORÉE ===")
# D- et L-alanine : même squelette, chiralité opposée
for name, smi in [("L-alanine", "C[C@@H](N)C(=O)O"), ("D-alanine", "C[C@H](N)C(=O)O")]:
    m = Chem.MolFromSmiles(smi)
    print(f"  {name:12s} {smi:22s} InChIKey complet {Chem.MolToInchiKey(m)}")
print(f"  -> InChIKey-14 identiques : {key14('C[C@@H](N)C(=O)O') == key14('C[C@H](N)C(=O)O')}")
print("     Rendre les deux énantiomères, c'est gâcher un slot pour zéro point.")

print("\n=== 2. plusieurs SMILES écrivent la même molécule ===")
for smi in ["c1ccccc1O", "OC1=CC=CC=C1", "Oc1ccccc1"]:
    print(f"  {smi:16s} -> {key14(smi)}")
print("     Le dédoublonnage doit se faire sur la clé, jamais sur la chaîne SMILES.")

In [ ]:
def mrr(predictions, truth, n_max=25):
    # MRR sur InChIKey-14. `predictions` : {molecule_id: [smiles, ...]} classés.
    scores = {}
    for mid, true_smi in truth.items():
        t = key14(true_smi)
        rank, seen = 0.0, []
        for smi in predictions.get(mid, [])[:n_max]:
            k = key14(smi)
            if k is None or k in seen:      # invalide ou doublon : ne consomme pas de slot utile
                continue
            seen.append(k)
            if k == t:
                rank = 1.0 / len(seen)
                break
        scores[mid] = rank
    return scores

# démonstration : la caféine est la bonne réponse pour m1 et m2
CAFEINE = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"
truth = {"m1": CAFEINE, "m2": CAFEINE, "m3": CAFEINE}
preds = {
    "m1": [CAFEINE, "CCO", "c1ccccc1"],                       # bon du premier coup
    "m2": ["CCO", "c1ccccc1", CAFEINE],                        # bon en 3e
    "m3": ["CCO", "CCO", "OCC", CAFEINE],                      # 3 doublons d'éthanol, puis le bon
}
s = mrr(preds, truth)
for mid, v in s.items():
    print(f"  {mid} : {v:.3f}")
print(f"\n  MRR = {np.mean(list(s.values())):.3f}")
print("\n  m3 illustre le dédoublonnage : 'CCO' et 'OCC' sont le même éthanol, donc le bon")
print("  candidat arrive en position UTILE 2 (0.500) et non 4 (0.250). C'est exactement ce")
print("  que fait la métrique officielle, et c'est pourquoi dédoublonner soi-même sur la clé")
print("  avant de soumettre ne gagne presque rien : le scorer le fait déjà.")

### Le détail qui fausse toute validation locale

La métrique officielle compare la clé **tautomère-canonique**, pas la clé brute. Un
tautomère est une molécule qui existe en plusieurs formes interconvertibles (un proton
qui se déplace) — même composé, squelette écrit différemment, donc InChIKey-14 différent.

`dariushafshar` a mesuré que **10,4 % des clés d'un pool de candidats changent** entre la
forme brute et la forme tautomère-canonique. Un MRR calculé en local sur les clés brutes
est donc **sous-estimé d'environ 0.007** — plus que ce que valent la plupart des étapes
publiques du leaderboard.

Si on construit une validation locale, elle doit canoniser les tautomères comme le scorer,
sinon elle mesure autre chose. C'est noté ici parce que c'est le genre de détail qui fait
qu'on « gagne » 0.005 en local et rien sur le board.

---
# 2. La data

## 2.1 Trois fichiers, onze bibliothèques

In [ ]:
for f in ["train.parquet", "test.parquet"]:
    p = pq.ParquetFile(RAW / f)
    print(f"{f:16s} {p.metadata.num_rows:>10,} lignes   {len(p.schema_arrow):2d} colonnes   "
          f"{(RAW/f).stat().st_size/1e9:.2f} GB")

print("\n=== colonnes de train absentes de test ===")
ctr = {f.name for f in pq.ParquetFile(RAW/"train.parquet").schema_arrow}
cte = {f.name for f in pq.ParquetFile(RAW/"test.parquet").schema_arrow}
print(f"  étiquettes (attendu) : {sorted(ctr - cte - {'precursor_error_ppm','num_peaks','base_peak_intensity','adduct_orig','ingest_lib'})}")
print(f"  autres               : {sorted((ctr - cte) & {'precursor_error_ppm','num_peaks','base_peak_intensity','adduct_orig','ingest_lib'})}")
print("\n  !! molecular_formula est dans train et PAS dans test. Il faut la déduire du")
print("     precursor_mz et de l'adduit — et c'est au sein d'une même formule que se joue")
print("     l'essentiel de la difficulté (§2.7).")

In [ ]:
meta = pd.read_parquet(RAW / "train.parquet", columns=[
    "ingest_lib", "inchikey14", "molecular_formula", "adduct",
    "instrument_type", "ionization_mode", "num_peaks", "precursor_mz"])

lib = (meta.groupby("ingest_lib")
       .agg(spectres=("inchikey14", "size"), molecules=("inchikey14", "nunique"),
            pics_median=("num_peaks", "median"))
       .sort_values("spectres", ascending=False))
lib["spec_par_mol"] = (lib.spectres / lib.molecules).round(1)
lib["part_%"] = (lib.spectres / len(meta) * 100).round(1)
lib["timsTOF_%"] = (meta.assign(t=meta.instrument_type.eq("timsTOF"))
                    .groupby("ingest_lib").t.mean().mul(100).round(1))
print(lib.to_string())
print(f"\nTOTAL {len(meta):,} spectres, {meta.inchikey14.nunique():,} molécules uniques")

## 2.2 L'asymétrie : le test est un autre instrument

La dernière colonne ci-dessus est la chose la plus importante de ce notebook.

`timsTOF` est un type de spectromètre (Bruker, à mobilité ionique couplée au temps de
vol). Les deux bibliothèques `enveda-*` en sont à 100 %. **Toutes les autres sont à 0 %.**
Et le jeu de test ?

In [ ]:
print("=== instrument_type dans test.parquet ===")
print(test.instrument_type.value_counts().to_string())

t = meta[meta.instrument_type == "timsTOF"]
print(f"\n=== timsTOF dans train ===")
print(f"{len(t):,} spectres sur {len(meta):,} ({len(t)/len(meta)*100:.1f} % du corpus)")
print(t.ingest_lib.value_counts().to_string())

print(f"\n=== densité de pics ===")
test_pk = test.ms2_mzs.apply(len)
print(f"  test                    médiane {test_pk.median():6.0f} pics")
print(f"  train timsTOF           médiane {t.num_peaks.median():6.0f} pics")
print(f"  train tout le reste     médiane {meta[meta.instrument_type != 'timsTOF'].num_peaks.median():6.0f} pics")
print("\nLe test est 100 % timsTOF, et plus dense encore que le timsTOF d'entraînement.")
print("Le tutoriel officiel exclut enveda-180 en une ligne, sans commentaire :")
print("    filters=[('ingest_lib', '!=', 'enveda-180')]")
print("...c'est-à-dire qu'il jette 45 % du corpus, et la seule partie qui partage")
print("l'instrument du test. Tout le terrain a hérité de cette ligne.")

In [ ]:
# FIGURE 2 — magnitude (densité de pics) + identité (instrument). Deux couleurs, pas onze.
d = lib.sort_values("pics_median")
is_tims = d["timsTOF_%"] > 50
colors = np.where(is_tims, ORANGE, BLUE)

fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(d))
ax.barh(y, d.pics_median.values, color=colors, height=0.62)
ax.axvline(test_pk.median(), color=INK, linewidth=2, linestyle=(0, (4, 3)), zorder=3)
ax.annotate(f"test : {test_pk.median():.0f} pics", xy=(test_pk.median(), len(d) - 0.4),
            xytext=(-8, 0), textcoords="offset points", ha="right", va="center",
            fontsize=9, fontweight="bold", color=INK)

for yi, v in zip(y, d.pics_median.values):           # étiquette directe sur chaque barre
    ax.annotate(f"{v:.0f}", xy=(v, yi), xytext=(4, 0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK2)

ax.set_yticks(y); ax.set_yticklabels(d.index, fontsize=9)
ax.set_xlabel("nombre de pics par spectre (médiane)")
ax.set_title("Densité des spectres par bibliothèque", loc="left", color=INK)
ax.grid(axis="y", visible=False); ax.set_xlim(0, 260)

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ORANGE, label="timsTOF (= l'instrument du test)"),
                   Patch(color=BLUE,   label="autres instruments")],
          loc="lower right")
plt.tight_layout(); plt.show()

## 2.3 Le même composé, mesuré sur les deux

Un écart de médianes pourrait n'être qu'un effet de composition : peut-être qu'Enveda
mesure des molécules plus grosses. Le test honnête est de prendre **la même molécule, le
même adduit**, et de comparer les deux spectres.

2 288 molécules sont présentes dans `enveda-180` *et* dans une bibliothèque publique.

In [ ]:
env_mols = set(meta[meta.ingest_lib.isin(["enveda-180", "enveda-np-examples"])].inchikey14)
pub_mols = set(meta[~meta.ingest_lib.isin(["enveda-180", "enveda-np-examples"])].inchikey14)
shared   = env_mols & pub_mols
print(f"molécules enveda            : {len(env_mols):>8,}")
print(f"molécules publiques         : {len(pub_mols):>8,}")
print(f"présentes dans les DEUX     : {len(shared):>8,}  "
      f"({len(shared)/len(env_mols)*100:.1f} % d'enveda, {len(shared)/len(pub_mols)*100:.1f} % du public)")
print(f"union                       : {len(env_mols | pub_mols):>8,}")
print("\nLes bibliothèques ne diffèrent pas seulement par l'instrument : elles sont presque")
print("DISJOINTES en structures. Enveda couvre un espace chimique que le public ne couvre pas.")

In [ ]:
# un composé bien mesuré des deux côtés, pour la figure
PICK = "WUBXBUWVHIGIGM"
pair = pd.read_parquet(RAW / "train.parquet",
                       columns=["ingest_lib","inchikey14","adduct","normalized_smiles",
                                "precursor_mz","ms2_mzs","ms2_normalized_intensities","num_peaks"],
                       filters=[("inchikey14","==",PICK), ("adduct","==","[M+H]+")])
pair["src"] = np.where(pair.ingest_lib.str.startswith("enveda"), "enveda", "public")
top = pair.sort_values("num_peaks", ascending=False).groupby("src").head(1).set_index("src")
print(f"molécule {PICK} — SMILES {top.normalized_smiles.iloc[0]}")
print(top[["ingest_lib","num_peaks","precursor_mz"]].to_string())

In [ ]:
# FIGURE 3 — miroir : même molécule, même adduit, deux instruments.
fig, ax = plt.subplots(figsize=(9, 4.4))
for src, col, sign in [("enveda", ORANGE, 1), ("public", BLUE, -1)]:
    r = top.loc[src]
    mz = np.asarray(r.ms2_mzs); it = np.asarray(r.ms2_normalized_intensities, dtype=float)
    it = it / it.max()
    ax.vlines(mz, 0, sign * it, color=col, linewidth=0.7)
    ax.annotate(f"{r.ingest_lib} — {len(mz)} pics", xy=(0.012, 0.93 if sign > 0 else 0.07),
                xycoords="axes fraction", ha="left", va="center",
                fontsize=10, fontweight="bold", color=col)

ax.axhline(0, color=INK2, linewidth=1)
ax.set_xlabel("m/z"); ax.set_ylabel("intensité relative")
ax.set_yticks([-1, -0.5, 0, 0.5, 1]); ax.set_yticklabels(["1", ".5", "0", ".5", "1"])
ax.set_title(f"Le même composé ({PICK}, [M+H]+) sur deux instruments", loc="left", color=INK)
ax.set_xlim(0, float(top.precursor_mz.iloc[0]) * 1.05); ax.set_ylim(-1.3, 1.3)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

print("Même molécule, même adduit. Le spectre timsTOF est un continuum de raies fines ; le")
print("spectre public est un peigne clairsemé. Un cosinus entre les deux compare surtout")
print("l'absence de pics d'un côté à leur présence de l'autre — c'est précisément ce que")
print("fait tout le leaderboard, et on le quantifie en §3.2.")

## 2.4 Combien de spectres par molécule

In [ ]:
spm_tr = meta.groupby("inchikey14").size()
print(f"spectres par molécule (train) : médiane {spm_tr.median():.0f}, moyenne {spm_tr.mean():.1f}, max {spm_tr.max():,}")
print(f"  molécules avec un seul spectre : {(spm_tr == 1).sum():,} ({(spm_tr == 1).mean()*100:.1f} %)")
print(f"\nCeci interdit un découpage train/val aléatoire PAR SPECTRE : la même molécule se")
print(f"retrouverait des deux côtés et la validation serait fuitée. Il faut découper PAR")
print(f"MOLÉCULE (par inchikey14) — c'est ce que fait le tutoriel officiel, à juste titre.")

print("\n=== adduits : train vs test ===")
cmp = pd.concat([
    meta.adduct.value_counts(normalize=True).mul(100).rename("train_%"),
    test.adduct.value_counts(normalize=True).mul(100).rename("test_%")], axis=1)
print(cmp.sort_values("test_%", ascending=False).head(7).round(1).to_string())
print("\n[M+H]+ domine les deux. Noter [M+CH2O2-H]- (adduit formiate) à 2,6 % du test :")
print("il existe dans train mais est rare, et GLACIER — un des modèles du stack public —")
print("n'est utilisé QUE sur [M+H]+, donc ces spectres-là ne reçoivent aucun score GLACIER.")

## 2.5 L'espace de recherche : de combien de candidats parle-t-on ?

La première contrainte utilisable est la masse. Le `precursor_mz` donne la masse de l'ion ;
en retirant la contribution connue de l'adduit on obtient la **masse neutre** de la
molécule. Tout candidat dont la masse neutre diffère de plus de quelques ppm est éliminé.

Voyons ce que cette contrainte laisse.

In [ ]:
import re
MASSES = {"C":12.0,"H":1.0078250319,"N":14.0030740052,"O":15.9949146221,"S":31.97207069,
          "P":30.97376151,"Cl":34.96885271,"Br":78.9183376,"F":18.99840320,"I":126.904473,
          "Na":22.98976928,"K":38.96370649,"Si":27.9769265,"Se":79.9165218,"B":11.0093055,
          "As":74.9215942,"Fe":55.9349421,"D":2.0141017780}
_tok = re.compile(r"([A-Z][a-z]?)(\d*)")
def formula_mass(f):
    m = 0.0
    for el, n in _tok.findall(f):
        if el not in MASSES: return np.nan
        m += MASSES[el] * (int(n) if n else 1)
    return m

struct = meta.drop_duplicates("inchikey14")[["inchikey14","molecular_formula"]].copy()
fm = pd.Series({f: formula_mass(f) for f in struct.molecular_formula.unique()})
struct["neutral_mass"] = struct.molecular_formula.map(fm)
# trié une fois par masse : toutes les fenêtres se prennent ensuite par searchsorted
ssort  = struct.dropna(subset=["neutral_mass"]).sort_values("neutral_mass").reset_index(drop=True)
masses = ssort.neutral_mass.values
print(f"{len(struct):,} structures, dont {len(ssort):,} de masse calculable")

PROTON = 1.00727646
ELECTRON = 0.00054858
SHIFT = {"[M+H]+": PROTON, "[M-H]-": -PROTON,
         "[M+Na]+": 22.98976928 - ELECTRON, "[M+K]+": 38.96370649 - ELECTRON,
         "[M+NH4]+": 18.033823 - ELECTRON, "[M+CH2O2-H]-": 46.00547930 - PROTON,
         "[M+Cl]-": 34.96885271 + ELECTRON}
tm = test.assign(shift=test.adduct.map(SHIFT))
tm["neutral_mass"] = tm.precursor_mz - tm["shift"]
mol_mass = tm.dropna(subset=["neutral_mass"]).groupby("molecule_id").neutral_mass.median()

print(f"{len(mol_mass)} / {test.molecule_id.nunique()} molécules de test avec masse neutre déduite")
print(f"masse neutre : médiane {mol_mass.median():.1f} Da, de {mol_mass.min():.1f} à {mol_mass.max():.1f}")
print(f"\nLa plage est ÉTROITE — {mol_mass.min():.0f} à {mol_mass.max():.0f} Da. Le test n'est pas")
print(f"un échantillon quelconque de la chimie : c'est une bande de petites molécules, taille")
print(f"typique des produits naturels et des médicaments. Tout candidat hors de cette plage")
print(f"est inutile, ce qui élimine d'emblée "
      f"{(1 - ssort.neutral_mass.between(mol_mass.min(), mol_mass.max()).mean())*100:.0f} % des structures de train.")

rows = []
for ppm in (5, 10, 20):
    lo = np.searchsorted(masses, mol_mass.values * (1 - ppm/1e6))
    hi = np.searchsorted(masses, mol_mass.values * (1 + ppm/1e6))
    n = hi - lo
    rows.append({"tolérance": f"±{ppm} ppm", "médiane": int(np.median(n)),
                 "moyenne": round(n.mean(), 1), "max": int(n.max()),
                 "sans candidat": int((n == 0).sum())})
    if ppm == 5: n5 = n
print("\n=== candidats parmi les 275 810 structures de train, dans la fenêtre de masse ===")
print(pd.DataFrame(rows).to_string(index=False))
print("\nÀ ±5 ppm : médiane 54 candidats, et AUCUNE molécule de test sans candidat. La masse")
print("seule réduit donc 275 810 structures à ~54. Reste à en choisir une sur 54 — et c'est")
print("là que tout le travail se passe. (Avec COCONUT + PubChem, le pool réel se compte en")
print("milliers par molécule, pas en dizaines.)")

In [ ]:
# FIGURE 4 — distribution : combien de candidats par molécule de test, à ±5 ppm.
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(n5, bins=40, color=BLUE, edgecolor=SURFACE, linewidth=0.8)
med = np.median(n5)
ax.axvline(med, color=ORANGE, linewidth=2)
ax.annotate(f"médiane {med:.0f}", xy=(med, ax.get_ylim()[1]*0.9), xytext=(8, 0),
            textcoords="offset points", color=ORANGE, fontsize=9, fontweight="bold")
ax.set_xlabel("structures de train dans la fenêtre de masse (±5 ppm)")
ax.set_ylabel("molécules de test")
ax.set_title("Taille de l'espace de recherche après la seule contrainte de masse",
             loc="left", color=INK)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 2.6 Le vrai problème : les isomères

Une fois la masse fixée, il reste les **isomères** : des molécules de formule brute
identique mais d'arrangement différent. Elles ont exactement la même masse — la masse ne
les sépare donc *jamais*. Seule la fragmentation peut le faire.

C'est pour ça que tout le stack public reclasse les candidats **à l'intérieur des groupes
de même formule**. Mesurons l'ampleur du problème.

In [ ]:
per_formula = struct.groupby("molecular_formula").size()
print(f"structures uniques : {len(struct):,}")
print(f"formules uniques   : {len(per_formula):,}")
print(f"squelettes par formule : moyenne {per_formula.mean():.1f}, médiane {per_formula.median():.0f}, max {per_formula.max()}")

share = (struct.molecular_formula.map(per_formula) > 1).mean()
print(f"\n>>> {share*100:.1f} % des structures partagent leur formule avec au moins un autre squelette.")
print(f"\nles 5 formules les plus peuplées :")
for f, n in per_formula.sort_values(ascending=False).head(5).items():
    print(f"  {f:14s} {n:4d} squelettes distincts, même masse exacte")

print(f"\n=== et pour le test ? ===")
formulas = ssort.molecular_formula.values
hits = np.array([
    len(set(formulas[np.searchsorted(masses, m*(1-5e-6)):np.searchsorted(masses, m*(1+5e-6))]))
    for m in mol_mass.values])
print(f"formules distinctes dans la fenêtre ±5 ppm : médiane {np.median(hits):.0f}, max {hits.max()}")
print(f"-> donc ~{np.median(n5):.0f} candidats répartis sur ~{np.median(hits):.0f} formules :")
print(f"   l'essentiel des concurrents d'un bon candidat ont SA formule. Séparer les isomères")
print(f"   n'est pas un raffinement, c'est le problème central.")

In [ ]:
# FIGURE 5 — combien de squelettes partagent une formule (échelle log : la queue est longue).
vc = per_formula.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(vc.index[:40], vc.values[:40], color=BLUE, width=0.72, edgecolor=SURFACE, linewidth=0.6)
ax.bar([1], [vc.loc[1]], color=ORANGE, width=0.72)
ax.annotate(f"{vc.loc[1]:,} formules n'ont qu'un\nseul squelette — les faciles",
            xy=(1, vc.loc[1]), xytext=(14, -6), textcoords="offset points",
            fontsize=9, color=ORANGE, fontweight="bold", va="top")
ax.set_yscale("log")
ax.set_xlabel("nombre de squelettes distincts partageant une formule brute")
ax.set_ylabel("nombre de formules (log)")
ax.set_title("La plupart des formules sont ambiguës", loc="left", color=INK)
ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

---
# 3. Les solutions

## 3.0 Deux paradigmes opposés

Il y a deux manières de répondre « quelle molécule a produit ce spectre ? », et elles
n'ont presque rien en commun.

| | **Générer** (*de novo*) | **Retrouver** (*retrieval + ranking*) |
|---|---|---|
| principe | un modèle écrit un SMILES caractère par caractère | on énumère des candidats d'une base, on les classe |
| espace | toute la chimie imaginable (~10⁶⁰ molécules) | ce qui est catalogué (10⁵–10⁸) |
| peut trouver | une molécule jamais vue | seulement ce qui est dans la base |
| le tutoriel officiel | **oui, c'est ça** | non |
| le leaderboard | non (canal de repli marginal) | **oui, c'est ça** |

Ce décalage est le fait le plus important sur cette compétition : **le tutoriel officiel
enseigne le paradigme que le leaderboard n'utilise pas.** Un transformer
encodeur-de-spectre → décodeur-de-SMILES, entraîné sur 1,39 M de spectres, se fait battre
par du library matching avec un classifieur par-dessus.

La raison est dans les chiffres du §2 : les molécules du test font 246–439 Da et sont des
produits naturels ou apparentés. Cet espace **est catalogué** — COCONUT en recense ~700 k,
PubChem ~10⁸. Générer de novo, c'est renoncer à cette information gratuite. Et la métrique
ne récompense pas la créativité : elle demande la bonne clé au rang 1.

Les quatre familles qui suivent sont les briques réellement utilisées.

## 3.1 Famille 1 — appariement spectral (*library matching*)

**L'idée, en une phrase** : si ce spectre ressemble à un spectre de référence dont on
connaît la molécule, c'est probablement la même molécule.

C'est la méthode historique de la spectrométrie de masse, et elle n'a aucun paramètre
appris. On représente chaque spectre comme un vecteur creux (un seau par intervalle de
m/z, valeur = √intensité) et on calcule un cosinus.

Deux raffinements standards :

- **cosinus modifié** : apparie aussi les pics décalés de la différence des masses
  précurseurs. Deux analogues (une molécule et son dérivé méthylé, par exemple) partagent
  des fragments décalés d'un `CH2` ; le cosinus brut ne les voit pas, le modifié oui.
- **propagation d'analogues** : si le meilleur appariement est un analogue de masse
  décalée plutôt qu'un appariement exact, on propose la molécule *modifiée* en
  conséquence. C'est la racine de toute la lignée du leaderboard public
  (`prvsiyan/analog-propagation-casmi-2026-baseline`).

### Mesurons ce que coûte l'écart instrumental

On a les 2 288 molécules présentes dans les deux mondes (§2.3). D'où une expérience
propre : prendre un spectre timsTOF comme requête, et tenter de retrouver la bonne
molécule dans **deux bibliothèques de référence contenant exactement les mêmes molécules**
— l'une publique (autre instrument), l'autre Enveda (même instrument).

Toute différence de résultat ne peut venir que de l'instrument.

In [ ]:
spec = pd.read_parquet(RAW / "train.parquet",
        columns=["ingest_lib","inchikey14","ms2_mzs","ms2_normalized_intensities","precursor_mz"],
        filters=[("adduct","==","[M+H]+")])
spec["is_env"] = spec.ingest_lib.str.startswith("enveda")
sh = set(spec[spec.is_env].inchikey14) & set(spec[~spec.is_env].inchikey14)
spec = spec[spec.inchikey14.isin(sh)].reset_index(drop=True)
print(f"{len(spec):,} spectres [M+H]+ sur les {len(sh):,} molécules communes")

BIN, MAXMZ = 0.01, 1000.0
NBINS = int(MAXMZ / BIN)
def binned(df):
    # spectres -> matrice creuse L2-normalisée, un seau de 0.01 Da par colonne
    r, c, v = [], [], []
    for i, (mz, it) in enumerate(zip(df.ms2_mzs.values, df.ms2_normalized_intensities.values)):
        mz = np.asarray(mz); it = np.asarray(it, dtype=float)
        k = mz < MAXMZ
        r.append(np.full(k.sum(), i)); c.append((mz[k] / BIN).astype(np.int32))
        v.append(np.sqrt(it[k]))
    X = sp.csr_matrix((np.concatenate(v), (np.concatenate(r), np.concatenate(c))),
                      shape=(len(df), NBINS))
    n = np.sqrt(X.multiply(X).sum(1)).A.ravel(); n[n == 0] = 1
    return sp.diags(1 / n) @ X

env = spec[spec.is_env]
pub = spec[~spec.is_env]
query  = env.groupby("inchikey14", as_index=False).nth(0)     # requête : 1er spectre enveda
ref_env = env.groupby("inchikey14", as_index=False).nth(-1)   # réf même instrument : dernier
ref_pub = pub.groupby("inchikey14", as_index=False).nth(0)    # réf autre instrument

# une molécule dont la référence enveda SERAIT la requête ne prouverait rien : on l'écarte
multi = set(env.inchikey14.value_counts().loc[lambda s: s > 1].index)
keep = multi & set(ref_pub.inchikey14)
query, ref_env, ref_pub = [d[d.inchikey14.isin(keep)].sort_values("inchikey14").reset_index(drop=True)
                           for d in (query, ref_env, ref_pub)]
assert (query.inchikey14.values == ref_env.inchikey14.values).all()
assert (query.inchikey14.values == ref_pub.inchikey14.values).all()
print(f"=> {len(query)} requêtes, et deux bibliothèques de {len(query)} molécules identiques")

In [ ]:
Q = binned(query)
res = {}
for label, ref in [("références publiques\n(autre instrument)", ref_pub),
                   ("références Enveda\n(même instrument)", ref_env)]:
    S = (Q @ binned(ref).T).toarray()
    diag = S.diagonal().copy()                      # score de la BONNE molécule
    rank = (S > diag[:, None]).sum(1) + 1           # rang de la bonne molécule
    res[label] = {"top1": (rank == 1).mean(), "top5": (rank <= 5).mean(), "mrr": (1/rank).mean()}
    print(f"{label.replace(chr(10),' '):42s} top-1 {res[label]['top1']*100:5.1f} %   "
          f"top-5 {res[label]['top5']*100:5.1f} %   MRR {res[label]['mrr']:.3f}")

g = res["références Enveda\n(même instrument)"]["top1"] - res["références publiques\n(autre instrument)"]["top1"]
print(f"\n>>> écart top-1 : {g*100:+.1f} points, uniquement dû à l'instrument de référence.")

In [ ]:
# FIGURE 6 — deux conditions, deux mesures (toutes deux dans [0,1] : un seul axe).
labels = list(res)
metrics = [("top-1", "top1"), ("MRR", "mrr")]
x = np.arange(len(metrics)); w = 0.34

fig, ax = plt.subplots(figsize=(7.2, 3.6))
for i, (lab, col) in enumerate(zip(labels, [BLUE, ORANGE])):
    vals = [res[lab][k] for _, k in metrics]
    b = ax.bar(x + (i - 0.5) * (w + 0.02), vals, w, color=col,
               label=lab.replace("\n", " "), edgecolor=SURFACE, linewidth=1.5)
    for rect, v in zip(b, vals):                      # étiquette directe, pas de lecture d'axe
        ax.annotate(f"{v:.3f}", xy=(rect.get_x() + rect.get_width()/2, v), xytext=(0, 3),
                    textcoords="offset points", ha="center", fontsize=9, color=INK2)

ax.set_xticks(x); ax.set_xticklabels([m for m, _ in metrics])
ax.set_ylim(0, 1); ax.set_ylabel("score")
ax.set_title(f"Retrouver la bonne molécule parmi {len(query)}, par cosinus spectral",
             loc="left", color=INK)
ax.grid(axis="x", visible=False); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()
del spec, Q

**Comment lire ce résultat — et comment ne pas le surinterpréter.**

Ce qui est établi : à molécules identiques, apparier un spectre timsTOF contre des
références timsTOF bat l'appariement contre des références d'autres instruments, de
quelques points de top-1. La comparaison est équitable — mêmes requêtes, mêmes molécules
candidates, seule la provenance des références change.

Ce qui **n'est pas** établi :

- Les scores absolus sont flatteurs. C'est une tâche à ~1 200 candidats avec la **bonne
  réponse garantie présente**. La vraie tâche a des milliers de candidats et aucune
  garantie — d'où un leaderboard à 0.50 et non à 0.80.
- Les 2 288 molécules communes ne sont pas représentatives : ce sont précisément les
  composés assez documentés pour figurer dans deux bibliothèques, donc les plus faciles.
- Un cosinus binné brut n'est pas un cosinus modifié avec tolérance en ppm. Le stack
  public fait mieux que ça.

C'est un signal directionnel, pas un chiffrage. Mais il va dans le sens de §2.2–2.3, et il
porte sur les 45 % du corpus que tout le monde jette.

## 3.2 Famille 2 — prédiction d'empreintes (le paradigme CSI:FingerID)

**Le problème du library matching** : il ne trouve que ce qui est *déjà dans la
bibliothèque de spectres*. Or on connaît des millions de structures et seulement ~10⁵
spectres. Comment classer un candidat dont on n'a aucun spectre ?

**La solution** : ne pas comparer spectre à spectre, mais passer par un intermédiaire
calculable des deux côtés — une **empreinte moléculaire** (*fingerprint*).

1. Un réseau apprend `spectre → empreinte` : un vecteur binaire de quelques milliers de
   bits où chaque bit dit « ce sous-motif chimique est présent ».
2. Pour **n'importe quelle** structure d'une base, on *calcule* l'empreinte exacte avec
   RDKit — gratuitement, sans spectre.
3. On classe les candidats par similarité (Tanimoto) entre l'empreinte prédite et
   l'empreinte calculée.

C'est l'idée de CSI:FingerID, et les `FPNet` du leaderboard public en sont des
variantes. L'empreinte est le pivot : elle découple la mesure de la base de structures.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator, DataStructs

fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def fp(smi):
    m = Chem.MolFromSmiles(smi)
    return None if m is None else fpgen.GetFingerprint(m)

# le côté « gratuit » du pivot : l'empreinte se calcule pour toute structure
demo = {"caféine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
        "théophylline (caféine moins un méthyle)": "Cn1c(=O)c2[nH]cnc2n(C)c1=O",
        "théobromine (autre isomère)": "Cn1cnc2c1c(=O)[nH]c(=O)n2C",
        "aspirine (sans rapport)": "CC(=O)Oc1ccccc1C(=O)O"}
ref = fp(demo["caféine"])
print("Tanimoto à la caféine, empreintes Morgan r=2, 2048 bits :\n")
for name, smi in demo.items():
    print(f"  {DataStructs.TanimotoSimilarity(ref, fp(smi)):.3f}   {name}")

print("\nL'empreinte ordonne correctement : les analogues proches scorent haut, la molécule")
print("sans rapport s'effondre. Un modèle qui prédit cette empreinte à partir du SEUL")
print("spectre peut donc classer des millions de candidats dont il n'a jamais vu le spectre.")
print("\nC'est aussi la limite : théophylline et théobromine sont des isomères de formule")
print(f"identique et se ressemblent trop ({DataStructs.TanimotoSimilarity(fp(demo['théophylline (caféine moins un méthyle)']), fp(demo['théobromine (autre isomère)'])):.3f})")
print("pour qu'une empreinte un peu bruitée les sépare. Le §2.6 disait que 90,6 % des")
print("structures sont dans ce cas — d'où la famille 3.")

## 3.3 Famille 3 — simulation directe (ICEBERG, GLACIER)

**L'idée** : prendre le problème à l'envers. Plutôt que de deviner la structure depuis le
spectre, **prédire le spectre de chaque candidat** et garder celui dont le spectre prédit
ressemble le plus au spectre mesuré.

C'est un *modèle direct* (`structure → spectre`), et c'est plus facile à apprendre que
l'inverse : la fragmentation suit une physique, on peut énumérer les coupures de liaisons
et apprendre lesquelles sont probables.

- **ICEBERG** modélise explicitement la fragmentation : il énumère les fragments
  accessibles et apprend l'intensité de chacun.
- **GLACIER** prédit le spectre plus directement, et est plus rapide.

Les deux sont entraînés sur MassSpecGym (version autorisée par les organisateurs ; les
variantes entraînées sur NIST sont interdites et peuvent faire **disqualifier**).

**Où ça sert exactement** : au reclassement **à l'intérieur des groupes de même formule
brute**. C'est le seul outil du lot qui attaque les isomères de front, parce qu'il regarde
*comment* la molécule casse et non seulement *de quoi* elle est faite. Le §2.6 explique
pourquoi c'est la pièce décisive.

Le coût : c'est lent. Le stack public ne simule que le top 40–80 candidats, après que les
autres familles ont fait le tri grossier.

> Détail opérationnel cher payé par le terrain : une incompatibilité de wheel RDKit
> (Python 3.12 vs 3.13) a **silencieusement mis à zéro** les scores ICEBERG/GLACIER
> pendant des jours. Plusieurs scores publics contradictoires viennent de là. Les
> notebooks sérieux épinglent `rdkit==2026.03.3`.

## 3.4 Famille 4 — génération *de novo*

C'est l'approche du tutoriel officiel, et l'architecture est classique :

```
spectre  ->  [plongement de pics]  ->  [encodeur Transformer]  ->  mémoire
                                                                     |
                           SMILES  <-  [décodeur Transformer autorégressif]
```

- Chaque pic `(m/z, intensité)` devient un vecteur via un plongement sinusoïdal sur le
  m/z — l'analogue du *positional encoding*, mais sur un axe de masse continu.
- L'encodeur (6 couches, 12 têtes, d=768) mélange les pics par self-attention.
- Le décodeur génère le SMILES token par token, conditionné sur la mémoire.
- À l'inférence, on échantillonne 25 candidats et on les classe par vraisemblance.

La même famille existe dans la littérature sous les noms MSNovelist, Spec2Mol, MassGenie.

**Pourquoi ça perd ici** : le modèle doit réinventer une chimie déjà catalographiée, et la
métrique ne demande pas de nouveauté — elle demande la bonne clé au rang 1. Sur 246–439 Da
de produits naturels, la base *contient* presque toujours la réponse.

**Où ça garde de la valeur** : comme *source de candidats*, pas comme réponse. Pour une
molécule que la base ne couvre pas, un générateur peut proposer une structure que le
ranker évaluera ensuite. La génération n'a alors besoin que de **rappel**, et la précision
reste au ranker. C'est l'angle D de [`../docs/public-meta.md`](../docs/public-meta.md),
largement inexploré — avec un avertissement mesuré : `prvsiyan` a montré qu'inonder le
pool de candidats sans filtre fait **tomber** le score de 0.335 à 0.205.

## 3.5 À côté : modèles de fondation spectraux

**DreaMS** est un encodeur de spectres pré-entraîné en auto-supervision sur de grandes
collections non étiquetées, qui produit un plongement par spectre. On s'en sert pour la
recherche par similarité dans l'espace latent plutôt que dans l'espace des pics — utile
quand les pics ne s'alignent pas (exactement notre problème d'instrument). Dans le stack
public il sert surtout à **remplir la queue** de la liste des 25, ce qui, d'après le §1.3,
ne vaut presque rien. Ses poids sont autorisés.

## 3.6 Ce que fait réellement le leaderboard

Les quatre familles ne sont pas en concurrence : le stack public les **empile**. Tel que
documenté étape par étape dans `notebooks/external/metric-audit-and-safe-tail` (0.433
vérifié, le meilleur score public confirmé) :

| étape | mécanisme | famille | gain revendiqué |
|---|---|---|---|
| 1 | pool de candidats : COCONUT + ChEBI/LIPID MAPS + structures de train (~711 k), plus un canal sur ~105 M de PubChem | — | — |
| 2 | preuves : appariement de bibliothèque, décalages d'adduits, propagation d'analogues, MetFrag-lite | 1 | — |
| 3 | empreintes prédites (FPNet) comme traits | 2 | — |
| 4 | deux rankers LightGBM indépendants | — | — |
| 5 | a priori de popularité : `0.25·(log1p(#substances PubChem) + log1p(#articles PubMed))` | — | +0.010 |
| 6 | **jointure PubChem** : classer les candidats PubChem *dans le vrai moteur* au lieu de les insérer à des slots fixes | — | **+0.011** |
| 7 | reclassement ICEBERG + GLACIER dans les groupes de même formule ; GLACIER sur `[M+H]+` seulement | 3 | +0.006 |
| 8 | fusion par rang réciproque des deux moteurs : `1/(3+r₁) + 0.6/(3+r₂)` | — | +0.016 |
| 9 | dédoublonnage sur l'identité de la métrique, remplissage de la queue | — | ≤0.002 |

L'étape 5 mérite un commentaire, parce qu'elle n'est pas de la chimie : **l'a priori de
popularité** dit qu'une molécule citée dans beaucoup d'articles est *a priori* plus
probable qu'une structure obscure de même masse. C'est vrai, et ça marche — mais c'est un
pari sur la composition du jeu de test, pas sur la physique. Si le test privé contient plus
de composés mal documentés que le public, ce terme se retourne.

## 3.7 Où est le score, et ce que ça implique

Trois faits à tenir ensemble :

1. **Seul le rang 1 compte vraiment** (§1.3). La queue vaut ≤0.002.
2. **Le test fait 400 molécules.** Une molécule vaut 0.0025. Les rangs 9 à 16 du
   leaderboard tiennent dans 0.004 — soit une ou deux molécules.
3. **Toutes les constantes du stack public** (`POP_MU 0.25`, `alpha 0.6`,
   `lib_max < 0.9`, top-40/60/80, slots 2/4/6/8/10…) ont été ajustées *par soumission*
   contre ces mêmes 400 molécules.

Donc une bonne part de l'écart entre notebooks publics est du bruit, et une bonne part du
réglage est du surapprentissage au leaderboard. Le cas d'école, mesuré par un concurrent :
verrouiller le rang 1 fait passer sa validation locale de 0.63 à **0.89**, et son score
public de 0.413 à **0.409**. Sa validation récompensait exactement ce que le test ne
contient pas.

## 3.8 Ce qu'on retient pour la suite

**La tâche.** Choisir une molécule parmi ~54 (train seul) à quelques milliers (avec
COCONUT/PubChem), dont ~90 % ont la même formule brute que la bonne réponse, et la mettre
au rang 1.

**La data.** 2,54 M de spectres, mais coupés en deux mondes : 45 % de timsTOF Enveda dense
(médiane 138 pics) et 55 % de bibliothèques publiques clairsemées (médiane 20), presque
disjointes en structures (2 288 molécules communes sur 273 k). **Le test est entièrement
du premier monde**, et le tutoriel officiel l'exclut en une ligne dont tout le terrain a
hérité.

**Les solutions.** Retrieval + ranking, pas de la génération. Quatre briques empilées :
appariement spectral, empreintes prédites, simulation directe pour les isomères, et un
ranker par-dessus. Plus un a priori de popularité qui est un pari sociologique déguisé.

**Ce qui est faible chez les concurrents** : la mesure. Trois nombres vérifiés dans tout
le record public, et 400 molécules pour arbitrer.

**Ce qui suit, concrètement** : un découpage de validation **par molécule** et
**restreint au timsTOF**, donc fidèle au test et bien plus grand que les 400 molécules du
leaderboard public. C'est le préalable à toute amélioration qu'on voudra croire —
et l'expérience du §3.1 est le premier fil à tirer.